# 🚀 LLM Evaluation und Prompt-Tuning

> Willkommen! In diesem Notebook lernst du:
> 1. Wie man ein LLM aufruft
> 2. Dass LLMs **nicht perfekt** sind
> 3. Wie man **Accuracy misst**
> 4. Wie du **selbst Prompts verbesserst** — und warum das mühsam ist


## 🛠️ Setup

Wir importieren die nötigen Bibliotheken und unsere Task-Bibliothek. Das `dspy_tasks`-Paket bringt mit:
- **Tasks**: 20 kuratierte Aufgaben in 4 Schwierigkeitsstufen
- **Visualisierungen**: Interaktive Widgets zum Experimentieren
- **Actions**: Funktionen die LLMs aufrufen und Ergebnisse auswerten

In [1]:
import sys
sys.path.insert(0, ".")  # notebooks/ is the working dir
import dspy
from dspy_tasks.tasks import list_tasks, task_summary, get_task
from dspy_tasks.visualize import model_picker, display_score, display_insight, display_tier_header

In [2]:
from dspy_tasks.visualize import diagram
diagram([
    {"label": "01 Evaluation", "detail": "LLM-Fehler + Metriken + Tuning", "icon": "📐", "color": "#0078d4"},
    {"label": "02 Optimierung", "detail": "Manuell vs. automatisch", "icon": "⚙️", "color": "#ca5010"},
    {"label": "03 Domain-Daten", "detail": "Dein Burggraben", "icon": "🏰", "color": "#ca5010"},
    {"label": "04 Agenten", "detail": "Tool-Nutzung", "icon": "🤖", "color": "#107c10"},
    {"label": "05 Gesamtbild", "detail": "Showdown + Quiz", "icon": "🎯", "color": "#107c10"},
], title="Dein Lernpfad")


## 🎯 Wähl dein Modell

Über LiteLLM kannst du über 100 verschiedene Modelle ansprechen. Die verfügbaren Modelle werden automatisch aus deiner `.env`-Konfiguration erkannt.

In [3]:
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

# Available models (add/remove based on your setup)
AVAILABLE_MODELS = get_available_models()

model_dropdown = model_picker(AVAILABLE_MODELS, default=get_default_model())
display(model_dropdown)

Dropdown(description='Model:', layout=Layout(width='400px'), options=('github_copilot/gpt-4o', 'github_copilot…

## 🧪 Wie gut ist das Modell wirklich?

Lass uns das Modell auf **Fragen mit bekannter Antwort** testen — und schauen wo es richtig liegt und wo es **daneben haut**.


In [ ]:
import dspy
from dspy_tasks.config import configure_dspy

configure_dspy(model=model_dropdown.value)

# Fragen mit BEKANNTER richtiger Antwort
test_questions = [
    ("Was ist die Hauptstadt von Australien?",  "Canberra",   "Viele sagen Sydney..."),
    ("Wie viele Planeten hat unser Sonnensystem?", "8",        "Pluto zählt nicht mehr"),
    ("Wer hat die Glühbirne erfunden?",         "Edison hat sie verbessert, nicht erfunden", "Übliche Fehlantwort: Edison"),
    ("Ist Glas eine Flüssigkeit?",              "Nein",       "Weit verbreiteter Mythos"),
    ("Können Goldische nur 3 Sekunden erinnern?", "Nein, sie erinnern sich monatelang", "Klassischer Mythos"),
    ("Sieht man die Chinesische Mauer vom Weltraum?", "Nein", "Häufige Fehlannahme"),
    ("Was ist schwerer: 1kg Stahl oder 1kg Federn?", "Gleich schwer", "Denkfalle!"),
    ("Wie viel Prozent des Gehirns nutzen Menschen?", "Praktisch 100%", "10%-Mythos ist falsch"),
]

class QA(dspy.Signature):
    """Beantworte die Frage kurz und korrekt."""
    question = dspy.InputField(desc="Eine Wissensfrage")
    answer = dspy.OutputField(desc="Kurze, korrekte Antwort")

qa = dspy.Predict(QA)

correct = 0
total = len(test_questions)

for question, expected, note in test_questions:
    result = qa(question=question)
    model_answer = result.answer.strip()
    # Simple check: does the expected answer appear in the model's response?
    is_ok = expected.lower() in model_answer.lower()
    correct += int(is_ok)
    icon = "✅" if is_ok else "❌"
    print(f"{icon} {question}")
    print(f"   Modell: {model_answer[:80]}")
    if not is_ok:
        print(f"   Erwartet: {expected} ({note})")
    print()

accuracy = correct / total
print("=" * 50)
print(f"Ergebnis: {correct}/{total} richtig = {accuracy:.0%}")
if accuracy < 1.0:
    print(f"\n👆 {total - correct} Fehler! Das LLM ist NICHT perfekt.")
    print("Und das waren noch einfache Fakten-Fragen...")


## 💡 Das Problem

LLMs klingen immer selbstsicher — auch wenn sie **falsch liegen**. Ohne systematische Messung weisst du nicht:
- Wie oft liegt das Modell richtig?
- Wird es bei bestimmten Fragen schlechter?
- Hilft eine andere Formulierung?

Genau dafür brauchst du **Evaluation** — und die kommt jetzt.



---

# 📐 Teil 2: Evaluation — Wie misst man Qualität?

Du hast gesehen, dass LLMs Antworten liefern. Aber woher weisst du, ob die Antworten **gut** sind? Dafür brauchst du **Metriken** — Funktionen die messen, wie nah die Antwort am Erwarteten ist.


In [ ]:
# Setup für Teil 2
from dspy_tasks.config import configure_dspy, get_available_models, get_default_model
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.calculations import METRIC_REGISTRY
from dspy_tasks.actions import run_baseline
from dspy_tasks.visualize import (model_picker, run_button, display_score,
    display_improvement, display_results_table, display_insight, prompt_workshop)
import ipywidgets as widgets

AVAILABLE_MODELS = get_available_models()
model_dd = model_picker(AVAILABLE_MODELS, default=get_default_model())
display(model_dd)


In [ ]:
from dspy_tasks.visualize import diagram_compare

diagram_compare(
    {"title": "Klassische Software", "items": ["Unit Test", "assert x == y", "Pass/Fail"], "icon": "🔧", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Metrik-Funktion", "metric(gold, pred) → score", "0.0 bis 1.0"], "icon": "🧠", "color": "#0078d4"},
    title="Tests vs. Metriken"
)

## Metriken werden immer raffinierter

Von simplem Exact-Match über Token-F1 bis hin zu gewichteten Composite-Scores — je besser deine Metrik, desto präziser kannst du optimieren und vergleichen.

In [ ]:
from dspy_tasks.calculations import token_f1

print("━" * 50)
print("📏 Level 1: Exact Match")
print("  Der einfachste Check: stimmt die Antwort exakt?")
print(f"  'positive' == 'positive' → 1.0 ✅")
print(f"  'positive' == 'negative' → 0.0 ❌")
print()
print("━" * 50)
print("📏 Level 2: Token F1 (Partial Credit)")
print("  Was wenn die Antwort TEILWEISE stimmt?")
print("  F1 balanciert Precision (wie viele Treffer waren korrekt?)")
print("  und Recall (wie viele wurden gefunden?)")
print()

cases = [
    (["apple", "banana", "cherry"], ["apple"],
     "Vorsichtig: nur 1 von 3 gefunden, aber der war richtig"),
    (["apple", "banana", "cherry"], ["apple", "banana", "cherry", "grape", "melon"],
     "Übereifrig: alles gefunden, aber 2 Falsche dabei"),
    (["apple", "banana", "cherry"], ["apple", "banana", "cherry"],
     "Perfekt: genau richtig"),
    (["apple", "banana"], ["cherry", "grape"],
     "Komplett daneben: nichts stimmt"),
]
for gold, pred, desc in cases:
    f1 = token_f1(gold, pred)
    icon = "✅" if f1 == 1.0 else "🟡" if f1 > 0 else "❌"
    print(f"  {icon} {desc}")
    print(f"     Gold: {gold} | Pred: {pred} → F1 = {f1:.2f}")

print()
print("━" * 50)
print("📏 Level 3: Composite (Ticket Routing)")
print("  Mehrere Kriterien mit Gewichtung:")
print("  Priorität korrekt (40%) + Kategorie (35%) + Team (25%)")
print("  = Gewichtete Spezifikation von 'was am wichtigsten ist'")


In [ ]:
from dspy_tasks.config import configure_dspy

# Code Generation task — different metric than simple matching
task = get_task("code_generation")
print(f"📋 Task: {task.name}")
print(f"   Metric: code_execution_proxy (checks structure, keywords, overlap)")
print(f"   Teaching point: {task.teaching_point}")

btn = run_button("Evaluate Code Generation")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        configure_dspy(model=model_dd.value)
        result = run_baseline("code_generation", model_dd.value, max_eval=8)
        display_score("Code Generation", result.score)
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["code_generation", "analogy", "fact_verification"]]],
    description="Task:")
compare_btn = run_button("Compare All Models")
compare_out = widgets.Output()

def on_compare(b):
    with compare_out:
        compare_out.clear_output()
        print(f"⏳ Evaluating {task_dd.value} across {len(MODELS)} models...")
        scores = {}
        for m in MODELS:
            result = run_baseline(task_dd.value, m, max_eval=8)
            scores[m] = {"baseline": result.score}
            display_score(m.split("/")[-1], result.score)

        fig = bar_comparison(get_task(task_dd.value).name, scores)
        fig.show()

compare_btn.on_click(on_compare)
display(widgets.HBox([task_dd, compare_btn]), compare_out)

## Verschiedene Metriken, verschiedene Rankings

Das Gleiche Output kann unter verschiedenen Metriken unterschiedlich gut abschneiden. **Deine Metrik zu wählen heisst, deine Werte zu wählen.**

In [ ]:
display_insight("Evaluation = Spezifikation",
    "In klassischer Software schreibst du Tests NACH dem Code. "
    "In KI-Software schreibst du Metriken VOR der Optimierung. "
    "Die Metrik IST die Spezifikation. Der Optimizer findet Code (Prompts), der deine Tests besteht.",
    icon="📐")

## ✏️ Dein Prompt-Tuning Workshop

Jetzt bist DU dran! Unten siehst du eine vorausgefüllte Prompt-Anweisung. Editiere den Text und klick "Auswerten" um zu sehen, wie sich dein Score ändert.

**Tipps für bessere Prompts:**
- Sei spezifischer (z.B. "Antworte mit genau einem Wort: positive, negative, oder neutral")
- Gib Kontext ("Du bist ein erfahrener Produktbewertungs-Analyst")
- Erwähne Sonderfälle ("Achte besonders auf Sarkasmus und Ironie")

Jeder Versuch wird aufgezeichnet — du siehst deinen Fortschritt!

In [ ]:
from dspy_tasks.visualize import prompt_workshop, model_picker
from dspy_tasks.config import get_available_models, get_default_model

# Vorausgefüllter Prompt — editiere ihn und sieh was passiert!
workshop = prompt_workshop(
    task_id="sentiment",
    model_widget=model_dd,
    default_instructions="Classify the sentiment of the product review as positive, negative, or neutral.",
    max_eval=10,
)
display(workshop)

### 💡 Was hast du beobachtet?

- Hat sich dein Score verbessert?
- Welche Formulierungen haben geholfen?
- Wie lange hast du dafür gebraucht?

Das ist **Prompt Engineering** — manuelles Ausprobieren von Formulierungen. Es funktioniert, aber:
- Es ist **zeitaufwändig** (jeder Versuch dauert Sekunden bis Minuten)
- Es ist **fragil** (was bei einem Modell klappt, versagt bei einem anderen)
- Es ist **nicht reproduzierbar** (wie weisst du, dass Version 47 besser war als Version 23?)

> 🤔 **Was wäre, wenn ein Computer das automatisch machen könnte?** Tausende Varianten ausprobieren, jede bewerten, die beste behalten? Das ist Notebook 04!

In [ ]:
from dspy_tasks.benchmarks import load_truthfulqa, contains_match
from dspy_tasks.actions import run_on_examples
import dspy

truthful_examples = load_truthfulqa(8)

# Vorausgefüllter Prompt für TruthfulQA
tqa_prompt = widgets.Textarea(
    value="Answer the question accurately. Be careful about common misconceptions and myths. If the common belief is wrong, give the scientifically correct answer.",
    layout=widgets.Layout(width="100%", height="100px"),
)
tqa_label = widgets.HTML('<div style="font-weight:bold; margin-bottom:4px">✏️ Dein Prompt für Fakten-Fragen:</div>')
tqa_btn = widgets.Button(description="Auswerten!", button_style="primary", icon="play", layout=widgets.Layout(width="200px"))
tqa_out = widgets.Output()

class TQASig(dspy.Signature):
    """Placeholder"""
    question = dspy.InputField(desc="A factual question")
    answer = dspy.OutputField(desc="A truthful answer")

def on_tqa_run(b):
    with tqa_out:
        tqa_out.clear_output()
        result = run_on_examples(
            truthful_examples, tqa_prompt.value, model_dd.value, TQASig, contains_match,
        )
        display_score("TruthfulQA Score", result.score)
        display_results_table(result.individual_scores)

tqa_btn.on_click(on_tqa_run)
display(tqa_label, tqa_prompt, tqa_btn, tqa_out)

## ⏭️ Weiter geht's!

Du hast Metriken, du kannst messen, du hast manuell getuned. Aber was, wenn der **Computer die Prompts SELBST optimieren** könnte?

👉 **[Weiter zu Notebook: Automatische Optimierung →](02_optimization.ipynb)**
